In [ ]:
# 02 — Data Preprocessing
## Smart Retail Intelligence Platform

**Goal:** Clean, type-fix, encode, and time-split the sales data for forecasting.

**Input:**  `../../datasets/smart_retail_sales_dataset.csv`  
**Output:** `../../datasets/processed/clean_sales.csv`

**Steps:**
1. Load & basic checks  
2. Missing values  
3. Duplicates  
4. Invalid values  
5. Outliers (careful — retail has real high-value rows)  
6. Data types  
7. Categorical encoding  
8. Time-based train / val / test split  
9. Save cleaned dataset  

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
def resolve_path(*parts) -> Path:
    """Try common roots so notebook works from backend/notebooks/."""
    roots = [
        Path("../.."),          # backend/notebooks → project root
        Path(".."),
        Path("."),
        Path("/home/workdir/artifacts"),
    ]
    for root in roots:
        p = root.joinpath(*parts)
        if p.exists() or root.exists():
            return p.resolve() if p.exists() else (root / Path(*parts)).resolve()
    return Path(*parts)

RAW_PATH = None
for cand in [
    Path("../../datasets/smart_retail_sales_dataset.csv"),
    Path("../datasets/smart_retail_sales_dataset.csv"),
    Path("datasets/smart_retail_sales_dataset.csv"),
]:
    if cand.exists():
        RAW_PATH = cand.resolve()
        break

if RAW_PATH is None:
    raise FileNotFoundError("smart_retail_sales_dataset.csv not found under datasets/")

PROCESSED_DIR = Path("../../datasets/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Raw data : {RAW_PATH}")
print(f"📂 Output   : {PROCESSED_DIR}")

📂 Raw data : C:\Users\ASUS\OneDrive\Desktop\claude-test\smart-retail-intelligence-platform\datasets\smart_retail_sales_dataset.csv
📂 Output   : ..\..\datasets\processed


In [4]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Loaded: 300,000 rows × 34 columns


,sale_id,invoice_number,date,store_id,store_name,city,state,product_id,sku,product_name,category,brand,supplier,cost_price,selling_price,quantity_sold,revenue,profit,current_stock,reorder_level,promotion,discount_percent,holiday,festival,day_of_week,week_of_year,month,quarter,year,season,is_weekend,weather,customer_type,payment_method
0,1,INV-2021-97240,2021-01-01,1,SmartRetail Mumbai,Mumbai,Maharashtra,129,SKU-BOO-0129,DailyFresh Book 129,Books,DailyFresh,Supplier_West,623.10,873.21,3,"2,619.63",750.33,130,41,NaN,0.00,NaN,NaN,Friday,53,1,1,2021,Winter,0,Foggy,Member,Credit Card
1,2,INV-2021-32792,2021-01-01,1,SmartRetail Mumbai,Mumbai,Maharashtra,84,SKU-CLO-0084,DailyFresh Clothing 84,Clothing,DailyFresh,Supplier_North,"2,082.53","2,848.82",4,"11,395.28","3,065.16",121,11,NaN,0.00,NaN,NaN,Friday,53,1,1,2021,Winter,0,Clear,Regular,UPI
2,3,INV-2021-32792,2021-01-01,1,SmartRetail Mumbai,Mumbai,Maharashtra,195,SKU-SPO-0195,DailyFresh Sport 195,Sports,DailyFresh,Supplier_East,"3,158.10","4,812.53",8,"34,650.22","9,385.42",111,41,10% Off,0.10,NaN,NaN,Friday,53,1,1,2021,Winter,0,Cold,Regular,Debit Card


In [5]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Drop rows with invalid dates (should be rare)
bad_dates = df["date"].isna().sum()
if bad_dates:
    print(f"⚠️ Dropping {bad_dates} rows with invalid dates")
    df = df.dropna(subset=["date"]).copy()

df = df.sort_values(["date", "sale_id"]).reset_index(drop=True)

print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Rows after date clean: {len(df):,}")

Date range: 2021-01-01 → 2023-12-31
Rows after date clean: 300,000


In [6]:
print("Missing (NaN) before:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# promotion / holiday / festival: treat NaN + string "None" as "None"
for col in ["promotion", "holiday", "festival"]:
    if col in df.columns:
        df[col] = df[col].fillna("None").astype(str)
        df.loc[df[col].str.strip().str.lower().isin(
            ["none", "nan", "null", "", "na", "n/a"]
        ), col] = "None"

# Any leftover numeric NaNs → median (safe default)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if df[col].isna().any():
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f"  Filled {col} NaN with median={med:.2f}")

print("\nMissing after:")
left = df.isnull().sum()[df.isnull().sum() > 0]
print(left if len(left) else "✅ No missing values")

Missing (NaN) before:
promotion    148038
holiday      295933
festival     295933
dtype: int64

Missing after:
✅ No missing values


In [7]:
before = len(df)

# Exact full-row duplicates
n_exact = df.duplicated().sum()

# Same sale_id more than once (should be unique)
n_sale = df.duplicated(subset=["sale_id"]).sum() if "sale_id" in df.columns else 0

print(f"Exact duplicate rows : {n_exact:,}")
print(f"Duplicate sale_id    : {n_sale:,}")

df = df.drop_duplicates().copy()
if "sale_id" in df.columns:
    df = df.drop_duplicates(subset=["sale_id"], keep="first").copy()

print(f"Rows removed: {before - len(df):,}")
print(f"Rows left   : {len(df):,}")

Exact duplicate rows : 0
Duplicate sale_id    : 0
Rows removed: 0
Rows left   : 300,000


In [8]:
before = len(df)

# Impossible values → drop
mask_bad = (
    (df["quantity_sold"] <= 0) |
    (df["cost_price"] < 0) |
    (df["selling_price"] < 0) |
    (df["current_stock"] < 0)
)
n_bad = mask_bad.sum()
print(f"Rows with impossible values (qty<=0, negative price/stock): {n_bad:,}")
df = df.loc[~mask_bad].copy()

# discount_percent must be in [0, 1]
if "discount_percent" in df.columns:
    df["discount_percent"] = df["discount_percent"].clip(0, 1)

# selling_price < cost_price is allowed (clearance / BOGO) — keep, just flag
df["is_loss_price"] = (df["selling_price"] < df["cost_price"]).astype(int)
print(f"Rows with selling < cost (kept, flagged): {df['is_loss_price'].sum():,}")

print(f"Rows after invalid clean: {len(df):,} (removed {before - len(df):,})")

Rows with impossible values (qty<=0, negative price/stock): 0
Rows with selling < cost (kept, flagged): 0
Rows after invalid clean: 300,000 (removed 0)


In [9]:
# Retail has real high-ticket items (Electronics). We only cap extreme
# quantity_sold and revenue using a soft upper percentile, not strict IQR wipe.

def soft_cap(series, upper_q=0.999):
    """Cap only the top 0.1% — keeps real high-value sales."""
    hi = series.quantile(upper_q)
    n = (series > hi).sum()
    return series.clip(upper=hi), n, hi

df["quantity_sold"], n_q, hi_q = soft_cap(df["quantity_sold"])
df["revenue"], n_r, hi_r = soft_cap(df["revenue"])

print(f"quantity_sold capped at {hi_q:.0f}  ({n_q:,} rows touched)")
print(f"revenue capped at {hi_r:,.0f}     ({n_r:,} rows touched)")
print("Other numeric columns left as-is (domain-valid extremes).")

quantity_sold capped at 20  (277 rows touched)
revenue capped at 589,922     (296 rows touched)
Other numeric columns left as-is (domain-valid extremes).


In [10]:
# IDs as int
for col in ["sale_id", "store_id", "product_id", "year", "month", "quarter",
            "week_of_year", "is_weekend", "reorder_level", "current_stock",
            "quantity_sold"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Floats
for col in ["cost_price", "selling_price", "revenue", "profit", "discount_percent"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(float)

# Categories as string (clean)
cat_cols = [
    "store_name", "city", "state", "sku", "product_name", "category",
    "brand", "supplier", "promotion", "holiday", "festival",
    "day_of_week", "season", "weather", "customer_type", "payment_method",
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("dtypes after fix:")
print(df.dtypes)

dtypes after fix:
sale_id                      int64
invoice_number              object
date                datetime64[ns]
store_id                     int64
store_name                  object
city                        object
state                       object
product_id                   int64
sku                         object
product_name                object
category                    object
brand                       object
supplier                    object
cost_price                 float64
selling_price              float64
quantity_sold                int64
revenue                    float64
profit                     float64
current_stock                int64
reorder_level                int64
promotion                   object
discount_percent           float64
holiday                     object
festival                    object
day_of_week                 object
week_of_year                 int64
month                        int64
quarter                      int64
ye

In [11]:
# Binary flags
df["is_promo"] = (df["promotion"] != "None").astype(int)
df["is_holiday"] = (df["holiday"] != "None").astype(int)
df["is_festival"] = (df["festival"] != "None").astype(int)

# Profit margin (row-level)
df["profit_margin"] = np.where(
    df["revenue"] > 0,
    df["profit"] / df["revenue"],
    0.0,
)

# Stock pressure
df["stockout_flag"] = (df["current_stock"] <= df["reorder_level"]).astype(int)

# Calendar helpers (if not already present / consistent)
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["day_of_week_num"] = df["date"].dt.dayofweek          # 0=Mon … 6=Sun
df["is_weekend"] = (df["day_of_week_num"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

print("New helper columns:",
      ["is_promo", "is_holiday", "is_festival", "profit_margin",
       "stockout_flag", "day_of_week_num", "is_month_start", "is_month_end"])
df[["date", "is_promo", "is_festival", "stockout_flag", "profit_margin"]].head()

New helper columns: ['is_promo', 'is_holiday', 'is_festival', 'profit_margin', 'stockout_flag', 'day_of_week_num', 'is_month_start', 'is_month_end']


,date,is_promo,is_festival,stockout_flag,profit_margin
0,2021-01-01,0,0,0,0.29
1,2021-01-01,0,0,0,0.27
2,2021-01-01,1,0,0,0.27
3,2021-01-01,0,0,0,0.43
4,2021-01-01,1,0,0,-0.10


In [12]:
# Label-encode high-cardinality / nominal fields.
# Keep original string columns too (useful for reporting).
# Encoder mapping saved for inference later.

from sklearn.preprocessing import LabelEncoder

encode_cols = [
    "store_id",       # already numeric, but keep consistent
    "category",
    "brand",
    "supplier",
    "promotion",
    "holiday",
    "festival",
    "season",
    "weather",
    "customer_type",
    "payment_method",
    "city",
    "state",
]

# store_id is already int — skip re-encode if present
encode_cols = [c for c in encode_cols if c in df.columns and c != "store_id"]

label_encoders = {}
for col in encode_cols:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  {col:20s} → {col}_enc  ({df[col].nunique()} classes)")

# Save encoders for production inference
import pickle
enc_path = PROCESSED_DIR / "label_encoders.pkl"
with open(enc_path, "wb") as f:
    pickle.dump(label_encoders, f)
print(f"\n✅ Label encoders saved → {enc_path}")

  category             → category_enc  (8 classes)
  brand                → brand_enc  (8 classes)
  supplier             → supplier_enc  (5 classes)
  promotion            → promotion_enc  (5 classes)
  holiday              → holiday_enc  (6 classes)
  festival             → festival_enc  (6 classes)
  season               → season_enc  (4 classes)
  weather              → weather_enc  (9 classes)
  customer_type        → customer_type_enc  (3 classes)
  payment_method       → payment_method_enc  (4 classes)
  city                 → city_enc  (10 classes)
  state                → state_enc  (8 classes)

✅ Label encoders saved → ..\..\datasets\processed\label_encoders.pkl


In [13]:
# Label-encode high-cardinality / nominal fields.
# Keep original string columns too (useful for reporting).
# Encoder mapping saved for inference later.

from sklearn.preprocessing import LabelEncoder

encode_cols = [
    "store_id",       # already numeric, but keep consistent
    "category",
    "brand",
    "supplier",
    "promotion",
    "holiday",
    "festival",
    "season",
    "weather",
    "customer_type",
    "payment_method",
    "city",
    "state",
]

# store_id is already int — skip re-encode if present
encode_cols = [c for c in encode_cols if c in df.columns and c != "store_id"]

label_encoders = {}
for col in encode_cols:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(f"  {col:20s} → {col}_enc  ({df[col].nunique()} classes)")

# Save encoders for production inference
import pickle
enc_path = PROCESSED_DIR / "label_encoders.pkl"
with open(enc_path, "wb") as f:
    pickle.dump(label_encoders, f)
print(f"\n✅ Label encoders saved → {enc_path}")

  category             → category_enc  (8 classes)
  brand                → brand_enc  (8 classes)
  supplier             → supplier_enc  (5 classes)
  promotion            → promotion_enc  (5 classes)
  holiday              → holiday_enc  (6 classes)
  festival             → festival_enc  (6 classes)
  season               → season_enc  (4 classes)
  weather              → weather_enc  (9 classes)
  customer_type        → customer_type_enc  (3 classes)
  payment_method       → payment_method_enc  (4 classes)
  city                 → city_enc  (10 classes)
  state                → state_enc  (8 classes)

✅ Label encoders saved → ..\..\datasets\processed\label_encoders.pkl


In [14]:
# XGBoost / tree models do NOT need scaling.
# We still fit a scaler on train later if you use linear baselines.
# Here we only record which columns are scale-candidates.

scale_candidates = [
    "cost_price", "selling_price", "quantity_sold", "revenue", "profit",
    "current_stock", "reorder_level", "discount_percent", "profit_margin",
]
scale_candidates = [c for c in scale_candidates if c in df.columns]
print("Scale candidates (use only if needed):", scale_candidates)
print("Skipping fit here — scaler will be fit on TRAIN split only (no leakage).")

Scale candidates (use only if needed): ['cost_price', 'selling_price', 'quantity_sold', 'revenue', 'profit', 'current_stock', 'reorder_level', 'discount_percent', 'profit_margin']
Skipping fit here — scaler will be fit on TRAIN split only (no leakage).


In [15]:
# Time series rule: NEVER shuffle. Split by date.
# Default: 70% train | 15% val | 15% test  (by unique days)

dates = np.array(sorted(df["date"].dt.normalize().unique()))
n = len(dates)
i_train = int(n * 0.70)
i_val = int(n * 0.85)

train_end = dates[i_train - 1]
val_end = dates[i_val - 1]

df["split"] = "test"
df.loc[df["date"] <= train_end, "split"] = "train"
df.loc[(df["date"] > train_end) & (df["date"] <= val_end), "split"] = "val"

print("Split boundaries:")
print(f"  Train : … → {pd.Timestamp(train_end).date()}")
print(f"  Val   : {pd.Timestamp(train_end + pd.Timedelta(days=1)).date()} → {pd.Timestamp(val_end).date()}")
print(f"  Test  : {pd.Timestamp(val_end + pd.Timedelta(days=1)).date()} → …")
print()
print(df["split"].value_counts())
print()
print("Rows per split:")
display(df.groupby("split").agg(
    rows=("sale_id", "count"),
    days=("date", "nunique"),
    revenue=("revenue", "sum"),
    date_min=("date", "min"),
    date_max=("date", "max"),
))

Split boundaries:
  Train : … → 2023-02-05
  Val   : 2023-02-06 → 2023-07-19
  Test  : 2023-07-20 → …

split
train    209956
test      45065
val       44979
Name: count, dtype: int64

Rows per split:


,rows,days,revenue,date_min,date_max
split,,,,,
test,45065,165,"1,407,939,589.53",2023-07-20,2023-12-31
train,209956,766,"6,541,549,887.88",2021-01-01,2023-02-05
val,44979,164,"1,377,593,023.86",2023-02-06,2023-07-19


In [16]:
# Preferred column order (original + engineered)
preferred = [
    "sale_id", "invoice_number", "date", "split",
    "store_id", "store_name", "city", "state",
    "product_id", "sku", "product_name", "category", "brand", "supplier",
    "cost_price", "selling_price", "quantity_sold", "revenue", "profit",
    "profit_margin", "is_loss_price",
    "current_stock", "reorder_level", "stockout_flag",
    "promotion", "discount_percent", "is_promo",
    "holiday", "is_holiday", "festival", "is_festival",
    "day_of_week", "day_of_week_num", "week_of_year",
    "month", "quarter", "year", "season",
    "is_weekend", "is_month_start", "is_month_end",
    "weather", "customer_type", "payment_method",
]
# encoded columns
enc_cols = [c for c in df.columns if c.endswith("_enc")]
preferred = preferred + enc_cols

# keep only existing
cols = [c for c in preferred if c in df.columns]
# append any leftover columns
cols += [c for c in df.columns if c not in cols]
df = df[cols].copy()

print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Nulls left:", df.isnull().sum().sum())
df.head(3)

Final shape: 300,000 rows × 56 columns
Nulls left: 0


,sale_id,invoice_number,date,split,store_id,store_name,city,state,product_id,sku,product_name,category,brand,supplier,cost_price,selling_price,quantity_sold,revenue,profit,profit_margin,...,year,season,is_weekend,is_month_start,is_month_end,weather,customer_type,payment_method,category_enc,brand_enc,supplier_enc,promotion_enc,holiday_enc,festival_enc,season_enc,weather_enc,customer_type_enc,payment_method_enc,city_enc,state_enc
0,1,INV-2021-97240,2021-01-01,train,1,SmartRetail Mumbai,Mumbai,Maharashtra,129,SKU-BOO-0129,DailyFresh Book 129,Books,DailyFresh,Supplier_West,623.10,873.21,3,"2,619.63",750.33,0.29,...,2021,Winter,0,1,0,Foggy,Member,Credit Card,0,5,4,4,4,4,3,3,0,1,6,3
1,2,INV-2021-32792,2021-01-01,train,1,SmartRetail Mumbai,Mumbai,Maharashtra,84,SKU-CLO-0084,DailyFresh Clothing 84,Clothing,DailyFresh,Supplier_North,"2,082.53","2,848.82",4,"11,395.28","3,065.16",0.27,...,2021,Winter,0,1,0,Clear,Regular,UPI,1,5,2,4,4,4,3,0,2,3,6,3
2,3,INV-2021-32792,2021-01-01,train,1,SmartRetail Mumbai,Mumbai,Maharashtra,195,SKU-SPO-0195,DailyFresh Sport 195,Sports,DailyFresh,Supplier_East,"3,158.10","4,812.53",8,"34,650.22","9,385.42",0.27,...,2021,Winter,0,1,0,Cold,Regular,Debit Card,6,5,1,0,4,4,3,2,2,2,6,3


In [17]:
out_csv = PROCESSED_DIR / "clean_sales.csv"
df.to_csv(out_csv, index=False)
print(f"✅ Saved cleaned data → {out_csv}")
print(f"   Size: {out_csv.stat().st_size / (1024**2):.1f} MB")

# Also save split boundaries for later notebooks
meta = {
    "n_rows": int(len(df)),
    "n_cols": int(df.shape[1]),
    "date_min": str(df["date"].min().date()),
    "date_max": str(df["date"].max().date()),
    "train_end": str(pd.Timestamp(train_end).date()),
    "val_end": str(pd.Timestamp(val_end).date()),
    "split_counts": df["split"].value_counts().to_dict(),
    "encode_cols": encode_cols,
    "scale_candidates": scale_candidates,
}
with open(PROCESSED_DIR / "preprocess_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✅ Metadata → {PROCESSED_DIR / 'preprocess_meta.json'}")
print(f"✅ Encoders  → {PROCESSED_DIR / 'label_encoders.pkl'}")
print("\nNext → 03_feature_engineering.ipynb")
display(meta)

✅ Saved cleaned data → ..\..\datasets\processed\clean_sales.csv
   Size: 91.1 MB
✅ Metadata → ..\..\datasets\processed\preprocess_meta.json
✅ Encoders  → ..\..\datasets\processed\label_encoders.pkl

Next → 03_feature_engineering.ipynb


{'n_rows': 300000,
 'n_cols': 56,
 'date_min': '2021-01-01',
 'date_max': '2023-12-31',
 'train_end': '2023-02-05',
 'val_end': '2023-07-19',
 'split_counts': {'train': 209956, 'test': 45065, 'val': 44979},
 'encode_cols': ['category',
  'brand',
  'supplier',
  'promotion',
  'holiday',
  'festival',
  'season',
  'weather',
  'customer_type',
  'payment_method',
  'city',
  'state'],
 'scale_candidates': ['cost_price',
  'selling_price',
  'quantity_sold',
  'revenue',
  'profit',
  'current_stock',
  'reorder_level',
  'discount_percent',
  'profit_margin']}